In [1]:
import pandas as pd, numpy as np
from pathlib import Path

P = Path("/home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion_training/v3_rebuild/dataset_clean.parquet")
d = pd.read_parquet(P)
keys = [c for c in ("subject_id","task_name","task_file","window_idx","start_idx","end_idx","n_samples") if c in d.columns]
print("rows:", len(d), "| key columns present:", keys)

w = d[keys].copy()
print("\nn_samples value counts:")
print(w["n_samples"].value_counts().head())

rec = ["subject_id","task_file"] if "task_file" in w.columns else ["subject_id","task_name"]
per = w.groupby(rec).size()
print("\nrecordings:", len(per), "| windows per recording: min", per.min(), "median", int(per.median()), "max", per.max())

# Hop = spacing between consecutive window starts inside one recording.
hops, lens = [], []
for _, g in w.groupby(rec):
    g = g.sort_values("window_idx")
    s = g["start_idx"].to_numpy()
    if len(s) > 1:
        hops.append(np.diff(s))
    lens.append(g["n_samples"].to_numpy())
hops = np.concatenate(hops); lens = np.concatenate(lens)
print("\ndistinct hop values (samples):", sorted(set(hops.tolist()))[:10])
print("distinct n_samples values (samples):", sorted(set(lens.tolist()))[:10])
print("\nhop in samples: median", np.median(hops), "| window length in samples: median", np.median(lens))
print("ratio window/hop =", np.median(lens) / np.median(hops))

# Infer the sampling rate by testing candidate rates against a 60 s window.
for fs in (128, 256, 512, 1000):
    if np.median(lens) % fs == 0:
        print(f"  if fs={fs} Hz -> window {np.median(lens)/fs:.1f} s, hop {np.median(hops)/fs:.1f} s")


rows: 5640 | key columns present: ['subject_id', 'task_name', 'task_file', 'window_idx', 'start_idx', 'end_idx', 'n_samples']

n_samples value counts:
n_samples
15360    5640
Name: count, dtype: int64

recordings: 861 | windows per recording: min 1 median 6 max 49

distinct hop values (samples): [7680]
distinct n_samples values (samples): [15360]

hop in samples: median 7680.0 | window length in samples: median 15360.0
ratio window/hop = 2.0
  if fs=128 Hz -> window 120.0 s, hop 60.0 s
  if fs=256 Hz -> window 60.0 s, hop 30.0 s
  if fs=512 Hz -> window 30.0 s, hop 15.0 s
